In [1]:
import sys
sys.path.append("..")
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from src.data import load_clean
from src.features import add_distance_features

df = add_distance_features(load_clean())
print(df.shape)  # ожидаем 33769 строк

(33769, 97)


In [2]:
num_cols = ["accommodates", "bedrooms", "beds", "latitude", "longitude",
            "dist_colosseum_km", "dist_termini_km", "dist_st_peters_km",
            "dist_pantheon_km", "dist_trevi_km", "dist_nearest_landmark_km"]
cat_cols = ["room_type", "neighbourhood_cleansed"]

X = df[num_cols + cat_cols].copy()
for c in cat_cols:
    X[c] = X[c].astype("category")
y = df["log_price"]

In [3]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_base = np.zeros(len(df))
oof_lgb = np.zeros(len(df))

for tr, te in kf.split(df):
    train, test = df.iloc[tr], df.iloc[te]

    # бейзлайн: медиана log_price по (район, тип жилья)
    med = train.groupby(["neighbourhood_cleansed", "room_type"])["log_price"].median()
    keys = pd.MultiIndex.from_frame(test[["neighbourhood_cleansed", "room_type"]])
    oof_base[te] = med.reindex(keys).fillna(train["log_price"].median()).values

    # LightGBM
    model = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05,
                              num_leaves=31, random_state=42, verbose=-1)
    model.fit(X.iloc[tr], y.iloc[tr])
    oof_lgb[te] = model.predict(X.iloc[te])

In [4]:
is_centre = (df["neighbourhood_cleansed"] == "I Centro Storico").values

def report(name, pred_log):
    err = np.abs(np.expm1(pred_log) - df["price"].values)
    print(f"{name:10s} MAE all: {err.mean():6.1f} | centre: {err[is_centre].mean():6.1f} "
          f"| rest: {err[~is_centre].mean():6.1f} | R2(log): {r2_score(y, pred_log):.3f}")

report("baseline", oof_base)
report("lightgbm", oof_lgb)

baseline   MAE all:   86.7 | centre:  108.6 | rest:   63.0 | R2(log): 0.302
lightgbm   MAE all:   66.0 | centre:   79.1 | rest:   51.7 | R2(log): 0.578


In [5]:
model.fit(X, y)
(pd.Series(model.feature_importances_, index=X.columns)
   .sort_values(ascending=False))

latitude                    2061
longitude                   1564
dist_colosseum_km           1558
dist_st_peters_km           1314
dist_nearest_landmark_km    1301
dist_pantheon_km            1242
dist_termini_km             1195
accommodates                1152
dist_trevi_km               1135
beds                         934
bedrooms                     672
room_type                    533
neighbourhood_cleansed       339
dtype: int32